**Quantum Fan-out, the No-Cloning Theorem, and Boolean Satisfiability.**

**The fan-out question.**
In classical digital logic, fan-out is trivial: connect a wire to two
destinations and both carry the same bit.
The quantum analogue asks: given an unknown input qubit $|x\rangle$,
can we design a circuit whose output carries an independent copy of
$|x\rangle$ on *two separate output lines* simultaneously?
We probe this question with two gates: CSWAP and CCNOT, and we test
three input states for each: $|x\rangle=|0\rangle$, $|x\rangle=|1\rangle$,
and $|x\rangle=|{+}\rangle=\frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$.

**When fan-out appears to work.**
For the computational basis states $|0\rangle$ and $|1\rangle$,
both gates produce the same value of $|x\rangle$ on the $x$-output line
and on the $z$-output line.
Measuring either line gives the same definite result as the input.
This looks like successful fan-out, and for **basis** states it genuinely is.

**When fan-out fails.**
If $|x\rangle = |{+}\rangle$, both gates produce an *entangled* joint
output rather than two independent copies.
True fan-out of $|{+}\rangle$ would give all four outcomes
$(00, 01, 10, 11)$ at 25% each on the $(x\text{-out},\, z\text{-out})$ pair.
Instead only $00$ and $11$ appear, so the outputs are perfectly *correlated*,
meaning measuring one instantly determines the other.
That is entanglement, not independent copies.

**The no-cloning theorem.**
This failure is not a quirk of these two gates. It is a fundamental
law of quantum mechanics: there is no unitary $U$ such that
$U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle$ **for all** $|\psi\rangle$.
Fan-out succeeds only for states we already know ($|0\rangle$ or $|1\rangle$);
it fails for any superposition because copying a superposition would
require measuring it first, which destroys it.

---
**Propositional Calculus and SAT.**
In 1971, Stephen Cook proved that the Boolean satisfiability problem (SAT)
is **NP-complete** on classical computers.
SAT asks whether the variables of a Boolean formula can be assigned
TRUE or FALSE so the formula evaluates to TRUE.
For example:
- $(ab)(bc)(ac) = 1$ ✓: satisfiable ($a=b=c=1$ works)
- $(ab)(bc)(a\bar{c}) = 1$ ✗: unsatisfiable ($bc$ forces $c=1$
  but $a\bar{c}$ forces $c=0$: a contradiction)

**The quantum SAT idea.**
A quantum circuit places all input qubits in superposition simultaneously,
encoding all $2^n$ input combinations at once.
A tree of Toffoli gates evaluates the Boolean formula across all
combinations in a single execution.
Crucially, each input qubit ($a$, $b$, $c$) acts only as a *control*:
it passes through gates unchanged and is never copied.
Using a qubit as a control is categorically different from fan-out
and does not violate no-cloning.

**This notebook has three parts:**

1.) **CSWAP (Fredkin gate)**. Cells 02–04: fan-out of $|0\rangle$
    and $|1\rangle$ appears to succeed but `fails` for $|{+}\rangle$ input.

2.) **CCNOT (Toffoli gate)**. Cells 05–07: same result; fan-out
    works for basis states but produces `entanglement` for $|{+}\rangle$ input.

3.) **SAT circuits**. Cells 08–09: evaluates $(ab)(bc)(ac)=1$
    (satisfiable) and $(ab)(bc)(a\bar{c})=1$ (unsatisfiable), confirming
    the circuit returns zero probability when no satisfying assignment exists.

---
**Cell 01 - Imports and shared backend.**
A single `AerSimulator` backend is shared across all cells.
All circuits use the simulator default of 1024 shots.

In [ ]:
"""satisfiability.ipynb"""

# Cell 01 - Imports and shared backend

import qiskit
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_distribution
from qiskit_aer import AerSimulator

backend = AerSimulator()

print(f"Qiskit version : {qiskit.__version__}")
print(f"Backend        : {backend.name}")


---
**Cell 02 - CSWAP fan-out: basis state input $|x\rangle=|0\rangle$,
 input state $|x,y,z\rangle = |0,1,0\rangle$.**

The CSWAP (Fredkin) gate swaps $y$ and $z$ when the control $x=1$,
and leaves them unchanged when $x=0$.
We use ancilla $y=|1\rangle$ and $z=|0\rangle$ and ask only whether
the $x$-output line and the $z$-output line both carry the value of
$x_\text{in}$.
The $y$-output is ignored throughout.

With $|x\rangle=|0\rangle$ the swap condition is not met.
$z$ stays $|0\rangle$ and $x$ passes through unchanged:

$$|0,1,0\rangle \xrightarrow{\text{CSWAP}} |0,1,0\rangle$$

Both the $x$-output and $z$-output equal 0, matching $x_\text{in}=0$.
Fan-out appears to succeed for this input. ✓

In [ ]:
# Cell 02 - CSWAP fan-out: |x,y,z> = |0,1,0>
# x=0 so no swap occurs; z stays |0>.
# x-output = 0, z-output = 0 -> both match x_in = 0.
# We measure only q0 (x-output) and q2 (z-output); q1 (y-output) is ignored.

qc = QuantumCircuit(3, 2)
# q0 = x = |0> (default, no gate needed)
qc.x(1)  # q1 = y = |1> (ancilla)
# q2 = z = |0> (ancilla, default)

qc.cswap(0, 1, 2)  # CSWAP: swap y and z when x=1

qc.measure(0, 0)  # x-output -> classical bit 0
qc.measure(2, 1)  # z-output -> classical bit 1  (y-output q1 not measured)

result = backend.run(transpile(qc, backend), shots=1024).result()
counts = result.get_counts(qc)
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
print("Input |x>=|0>: x-out=0, z-out=0 -> both match x_in -> fan-out succeeded")


---
**Cell 03 - CSWAP fan-out: basis state input $|x\rangle=|1\rangle$,
 input state $|x,y,z\rangle = |1,1,0\rangle$.**

Now with $|x\rangle=|1\rangle$ the swap condition is met.
$y$ and $z$ exchange values:

$$|1,1,0\rangle \xrightarrow{\text{CSWAP}} |1,0,1\rangle$$

The $x$-output is 1 (unchanged control) and the $z$-output is 1
(received the value of $y$, which happened to equal $x_\text{in}$).
Both output lines carry the value of $x_\text{in}=1$.
Fan-out again appears to succeed. ✓

Note that $y$ was destroyed in this process (its output is 0).
This matters for other uses of the gate but is irrelevant to the
fan-out question, which concerns only the $x$ and $z$ output lines.

In [ ]:
# Cell 03 - CSWAP fan-out: |x,y,z> = |1,1,0>
# x=1 so swap occurs: y and z exchange. z-output receives y=1 = x_in.
# x-output = 1, z-output = 1 -> both match x_in = 1.
# We measure only q0 (x-output) and q2 (z-output); q1 (y-output) is ignored.

qc = QuantumCircuit(3, 2)
qc.x(0)  # q0 = x = |1>
qc.x(1)  # q1 = y = |1> (ancilla)
# q2 = z = |0> (ancilla, default)

qc.cswap(0, 1, 2)  # CSWAP: swap y and z when x=1

qc.measure(0, 0)  # x-output -> classical bit 0
qc.measure(2, 1)  # z-output -> classical bit 1  (y-output q1 not measured)

result = backend.run(transpile(qc, backend), shots=1024).result()
counts = result.get_counts(qc)
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
print("Input |x>=|1>: x-out=1, z-out=1 -> both match x_in -> fan-out succeeded")


---
**Cell 04 - CSWAP fan-out fails: superposition input
 $|x\rangle=|{+}\rangle$, input state
 $|x,y,z\rangle = |{+},1,0\rangle$.**

Now $|x\rangle=|{+}\rangle=\frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$.
The full 3-qubit input state is
$\frac{1}{\sqrt{2}}(|0,1,0\rangle+|1,1,0\rangle)$.
After CSWAP the joint output state is:

$$\frac{1}{\sqrt{2}}\bigl(|0,1,0\rangle + |1,0,1\rangle\bigr)$$

Tracing out the $y$-output and looking only at the $(x\text{-out}, z\text{-out})$
pair, the reduced state is:

$$\frac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr)$$

This is a maximally entangled Bell state.
True fan-out of $|{+}\rangle$ onto two *independent* output lines would
produce the product state
$|{+}\rangle\otimes|{+}\rangle = \frac{1}{2}(|00\rangle+|01\rangle+|10\rangle+|11\rangle)$,
giving all four outcomes at 25% each.
Instead only $|00\rangle$ and $|11\rangle$ appear at 50% each.
The outputs are perfectly correlated: measuring the $x$-output as 0
instantly forces the $z$-output to 0, and vice versa.
This is `entanglement`, where the two lines share a joint state that cannot
be factored into two independent single-qubit states.
Neither output holds an independent copy of $|{+}\rangle$.
Fan-out has **failed**. ✗

In [ ]:
# Cell 04 - CSWAP fan-out: |x,y,z> = |+,1,0>
#
# True fan-out of |+> onto independent x-out and z-out lines would give:
#   |+> ⊗ |+> = 1/2(|00>+|01>+|10>+|11>) -> all four outcomes at 25%
#
# Actual output on (x-out, z-out):
#   1/sqrt(2)(|00>+|11>) -> only 00 and 11 at 50% each (Bell state)
#
# The two output lines are ENTANGLED, not independent copies. Fan-out FAILS.

qc = QuantumCircuit(3, 2)
qc.h(0)  # q0 = x = |+> = (|0>+|1>)/sqrt(2)
qc.x(1)  # q1 = y = |1> (ancilla)
# q2 = z = |0> (ancilla, default)

qc.cswap(0, 1, 2)  # CSWAP: swap y and z when x=1

qc.measure(0, 0)  # x-output -> classical bit 0
qc.measure(2, 1)  # z-output -> classical bit 1

result = backend.run(transpile(qc, backend), shots=1024).result()
counts = result.get_counts(qc)
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
print("True fan-out: all four outcomes (00,01,10,11) at 25% each")
print("Actual output: only 00 and 11 at ~50% each -> ENTANGLED, not copied")
print("CSWAP fan-out FAILS for superposition input |x>=|+>")


---
**Cell 05 - CCNOT fan-out: basis state input $|x\rangle=|0\rangle$,
 input state $|x,y,z\rangle = |0,1,0\rangle$.**

The CCNOT (Toffoli) gate flips $z$ when *both* $x=1$ and $y=1$,
leaving $x$ and $y$ unchanged in all cases.
We use the same ancilla setup as the CSWAP experiments: $y=|1\rangle$,
$z=|0\rangle$, and we measure only the $x$-output and $z$-output.

With $|x\rangle=|0\rangle$: the condition $x=1 \wedge y=1$ is not met.
$z$ stays $|0\rangle$ and $x$ passes through unchanged:

$$|0,1,0\rangle \xrightarrow{\text{CCNOT}} |0,1,0\rangle$$

Both $x$-output and $z$-output equal 0, matching $x_\text{in}=0$.
Fan-out appears to succeed. ✓

In [ ]:
# Cell 05 - CCNOT fan-out: |x,y,z> = |0,1,0>
# x=0: flip condition (x=1 AND y=1) not met; z stays |0>.
# x-output = 0, z-output = 0 -> both match x_in = 0.
# We measure only q0 (x-output) and q2 (z-output); q1 (y-output) is ignored.

qc = QuantumCircuit(3, 2)
# q0 = x = |0> (default, no gate needed)
qc.x(1)  # q1 = y = |1> (ancilla, needed to enable the CCNOT)
# q2 = z = |0> (ancilla, default; will be flipped to |1> if x=1)

qc.ccx(0, 1, 2)  # CCNOT: flip z when x=1 AND y=1

qc.measure(0, 0)  # x-output -> classical bit 0
qc.measure(2, 1)  # z-output -> classical bit 1  (y-output q1 not measured)

result = backend.run(transpile(qc, backend), shots=1024).result()
counts = result.get_counts(qc)
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
print("Input |x>=|0>: x-out=0, z-out=0 -> both match x_in -> fan-out succeeded")


---
**Cell 06 - CCNOT fan-out: basis state input $|x\rangle=|1\rangle$,
 input state $|x,y,z\rangle = |1,1,0\rangle$.**

Now $|x\rangle=|1\rangle$: the condition $x=1 \wedge y=1$ is met.
$z$ flips from $|0\rangle$ to $|1\rangle$ while $x$ and $y$ are
both left unchanged:

$$|1,1,0\rangle \xrightarrow{\text{CCNOT}} |1,1,1\rangle$$

The $x$-output is 1 (unchanged control) and the $z$-output is 1
(flipped target).
Both output lines carry the value of $x_\text{in}=1$, and unlike
CSWAP the $y$ ancilla is also preserved.
Fan-out again appears to succeed. ✓

In [ ]:
# Cell 06 - CCNOT fan-out: |x,y,z> = |1,1,0>
# x=1: flip condition (x=1 AND y=1) met; z flips from |0> to |1>.
# x-output = 1, z-output = 1 -> both match x_in = 1.
# Note: unlike CSWAP, the y ancilla (q1) is also preserved at |1>.

qc = QuantumCircuit(3, 2)
qc.x(0)  # q0 = x = |1>
qc.x(1)  # q1 = y = |1> (ancilla)
# q2 = z = |0> (ancilla, default)

qc.ccx(0, 1, 2)  # CCNOT: flip z when x=1 AND y=1

qc.measure(0, 0)  # x-output -> classical bit 0
qc.measure(2, 1)  # z-output -> classical bit 1  (y-output q1 not measured)

result = backend.run(transpile(qc, backend), shots=1024).result()
counts = result.get_counts(qc)
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
print("Input |x>=|1>: x-out=1, z-out=1 -> both match x_in -> fan-out succeeded")


---
**Cell 07 - CCNOT fan-out fails: superposition input
 $|x\rangle=|{+}\rangle$\
 input state $|x,y,z\rangle = |{+},1,0\rangle$**

Now $|x\rangle=|{+}\rangle=\frac{1}{\sqrt{2}}(|0\rangle+|1\rangle)$.
The full 3-qubit input is
$\frac{1}{\sqrt{2}}(|0,1,0\rangle+|1,1,0\rangle)$.
After CCNOT the joint output state is:

$$\frac{1}{\sqrt{2}}\bigl(|0,1,0\rangle + |1,1,1\rangle\bigr)$$

Tracing out the $y$-output, the $(x\text{-out}, z\text{-out})$ pair is:

$$\frac{1}{\sqrt{2}}\bigl(|00\rangle + |11\rangle\bigr)$$

Identical to the CSWAP result: a maximally entangled Bell state.
Only $|00\rangle$ and $|11\rangle$ appear, not all four outcomes.
The outputs are perfectly correlated rather than independently distributed.
Neither output line holds an independent copy of $|{+}\rangle$.
Fan-out **fails** here too. ✗

In [ ]:
# Cell 07 - CCNOT fan-out: |x,y,z> = |+,1,0>
#
# True fan-out of |+> onto independent x-out and z-out lines would give:
#   |+> ⊗ |+> = 1/2(|00>+|01>+|10>+|11>) -> all four outcomes at 25%
#
# Actual output on (x-out, z-out):
#   1/sqrt(2)(|00>+|11>) -> only 00 and 11 at 50% each (Bell state)
#
# The two output lines are ENTANGLED, not independent copies. Fan-out FAILS.

qc = QuantumCircuit(3, 2)
qc.h(0)  # q0 = x = |+> = (|0>+|1>)/sqrt(2)
qc.x(1)  # q1 = y = |1> (ancilla)
# q2 = z = |0> (ancilla, default)

qc.ccx(0, 1, 2)  # CCNOT: flip z when x=1 AND y=1

qc.measure(0, 0)  # x-output -> classical bit 0
qc.measure(2, 1)  # z-output -> classical bit 1

result = backend.run(transpile(qc, backend), shots=1024).result()
counts = result.get_counts(qc)
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
print("True fan-out: all four outcomes (00,01,10,11) at 25% each")
print("Actual output: only 00 and 11 at ~50% each -> ENTANGLED, not copied")
print("CCNOT fan-out FAILS for superposition input |x>=|+>")


---
**Summary.**
Both CSWAP and CCNOT successfully copy $|x\rangle$ to the $z$-output
when $|x\rangle$ is a computational basis state.
Both fail when $|x\rangle$ is in superposition: the outputs become
entangled rather than independent.
This is the **no-cloning theorem** in action: no quantum gate can
fan out an arbitrary unknown qubit state to two independent output lines.

**Bridge to SAT.**
The SAT circuit uses $a$, $b$, $c$ in superposition as *controls* of
Toffoli gates. Never as targets to be copied.
A control qubit passes through unchanged; only the **ancilla target** flips.
This is not fan-out and does not violate no-cloning.


---
**Cell 08 - Boolean SAT: satisfiable formula $(ab)(bc)(ac) = 1$**

**Why test SAT on a quantum computer?**
A classical computer checking an $n$-variable SAT formula by brute force
must enumerate up to $2^n$ input combinations and test each one.
For $n=64$ variables that is $2^{64} \approx 1.8 \times 10^{19}$ tests;
more than a 18 billion billion (18 quintillion) which is far beyond
the reach of any classical machine.
A quantum circuit sidesteps this by placing all $n$ input qubits in
superposition simultaneously, encoding every combination at once, and
evaluating the formula across all $2^n$ inputs in a *single* $O(1)$
circuit execution regardless of $n$.

**The catch for SATISFIABLE formulas.**
When a formula is satisfiable, only the $k$ input combinations that make
it TRUE contribute amplitude to the output qubit's $|1\rangle$ state.
The probability of measuring 1 in any single shot is $k/2^n$.
If solutions are rare (which is what makes it a hard problem) that probability is tiny
and many shots are needed to detect satisfiability with confidence.
For our 3-variable formula the unique solution is $a=b=c=1$, so
$P(\text{output}=1) = 1/8 = 12.5\%$ and we expect to see a 1 roughly
once every 8 shots on average.
A basic SAT circuit alone therefore does not give a decisive single-shot
advantage for *detecting* `satisfiability`; Grover's algorithm can
amplify the probability quadratically, but that is a separate technique.

**The 8-qubit circuit uses:**
- $q_0, q_1, q_2$: input qubits $a$, $b$, $c$ (in superposition)
- $q_3$: ancilla for $(ab)$
- $q_4$: ancilla for $(bc)$
- $q_5$: ancilla for $(ac)$
- $q_6$: ancilla for $(ab)(bc)$
- $q_7$: output qubit: 1 if $(ab)(bc)(ac)=1$, else 0

Each input qubit acts only as a *control* of one or more Toffoli gates
and passes through the circuit unchanged.
This is not fan-out. The qubit is not copied to any output line.

In [ ]:
# Cell 08 - SAT: satisfiable formula (ab)(bc)(ac) = 1
# 8 qubits: q0=a, q1=b, q2=c (inputs in superposition),
#           q3=(ab), q4=(bc), q5=(ac), q6=(ab)(bc), q7=output

qc = QuantumCircuit(8, 1)

# Place all input qubits in superposition: encode all 8 (a,b,c) at once
qc.h(0)  # a
qc.h(1)  # b
qc.h(2)  # c

# Evaluate the three pairwise AND terms into ancilla qubits
qc.ccx(0, 1, 3)  # q3 = a AND b
qc.ccx(1, 2, 4)  # q4 = b AND c
qc.ccx(0, 2, 5)  # q5 = a AND c

# Combine: (ab)(bc) then AND with (ac) to get the full formula
qc.ccx(3, 4, 6)  # q6 = (ab) AND (bc)
qc.ccx(6, 5, 7)  # q7 = (ab)(bc) AND (ac) = formula result

qc.measure(7, 0)  # output qubit -> classical bit 0

# Here we use 1024 shots to get a good estimate of the distribution of outcomes
result = backend.run(transpile(qc, backend), shots=1024).result()
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
if "1" in result.get_counts(qc):
    print("Formula is SATISFIABLE")
else:
    print("Formula is UNSATISFIABLE")


---
**Cell 09 - Boolean SAT: unsatisfiable formula
$(ab)(bc)(\overline{a}c) = 1$**

**The structural asymmetry between the two cases.**

When a formula is unsatisfiable, $f(x)=0$ for every one of the $2^n$
input combinations.
The output qubit is $|0\rangle$ in every term of the superposition and
accumulates exactly zero amplitude on $|1\rangle$.
So the state the circuit prepares is deterministic:
$P(\text{output}=1)=0$ exactly, not approximately, as a structural
property of the quantum state.
When a formula is satisfiable with $k$ solutions, the output instead
carries amplitude on $|1\rangle$ and each shot returns 1 with
probability $k/2^n$.

**Why this is not a free single-shot decision procedure.**

It is tempting to conclude that one measurement settles it: measure 0
and declare "unsatisfiable".
But a single measured 0 does **not** prove unsatisfiability.
A satisfiable formula with rare solutions also returns 0 on almost every
shot, because $P(\text{output}=1)=k/2^n$ can be astronomically small.
One shot therefore cannot distinguish "exactly zero" from
"tiny but nonzero".
To separate the two cases from measurements alone you need many shots,
and sampling can never confirm that a probability is *exactly* zero
rather than merely below your resolution.
This is precisely why Grover's algorithm is needed: it amplifies a rare
satisfying amplitude quadratically, so a satisfiable formula becomes
reliably detectable in far fewer shots.

**What the quantum circuit does buy you.**

The genuine and immediate advantage is not in the measurement statistics
but in the *evaluation*: the circuit evaluates $f$ across all $2^n$ input
combinations in a **single** execution, with a gate count that grows only
polynomially with the formula size.
Classically, brute-force checking all $2^n$ assignments (the only general
way to prove unsatisfiability) requires up to $2^n$ evaluations.
For $n=64$ that is $2^{64}$ checks with no shortcut.
The quantum circuit performs the evaluation once; the remaining challenge
is reading the answer out, which is what Grover addresses.

We *can* confirm that this particular small circuit is unsatisfiable with
certainty, but only by a classical route: inspecting the exact output
amplitudes on a simulator (Cell 10).
That inspection is a teaching tool, not a speedup, because it costs
$O(2^n)$ classical work - exactly the exponential cost the quantum device
avoids at run time.

**The circuit change from Cell 08.**
The formula $(ab)(bc)(\overline{a}c)=1$ is unsatisfiable because the
$ab$ term requires $a=1$ while the $\overline{a}c$ term requires $a=0$.
The only change from Cell 08 is that $a$ is temporarily flipped to
$\overline{a}$ using an X gate *after* the $ab$ ancilla is already
computed into $q_3$, then restored before the combining step.
Because this formula has no satisfying assignment, the histogram below
shows only the outcome 0 on every shot - but keep in mind that a single 0
alone would not have been enough to prove it.

In [ ]:
# Cell 09 - SAT: unsatisfiable formula (ab)(bc)(NOT-a AND c) = 1
# The formula requires a=1 (from ab) and a=0 (from NOT-a AND c) simultaneously.
# No assignment satisfies both -> output qubit amplitude on |1> is exactly zero.

qc = QuantumCircuit(8, 1)

# Place all input qubits in superposition
qc.h(0)  # a
qc.h(1)  # b
qc.h(2)  # c

# Compute (ab) first, while a is still in its original state
qc.ccx(0, 1, 3)  # q3 = a AND b
qc.ccx(1, 2, 4)  # q4 = b AND c

# Temporarily flip a -> NOT-a to compute the (NOT-a AND c) term
qc.x(0)  # q0 = NOT-a
qc.ccx(0, 2, 5)  # q5 = NOT-a AND c
qc.x(0)  # restore q0 = a  (X gate is its own inverse)

# Combine terms using the same tree structure as Cell 08
qc.ccx(3, 4, 6)  # q6 = (ab) AND (bc)
qc.ccx(6, 5, 7)  # q7 = (ab)(bc) AND (NOT-a AND c) = formula result

qc.measure(7, 0)  # output qubit -> classical bit 0

result = backend.run(transpile(qc, backend), shots=1024).result()
display(qc.draw(output="mpl"))
display(plot_distribution(result.get_counts(qc)))
if "1" in result.get_counts(qc):
    print("Formula is SATISFIABLE")
else:
    print("Formula is UNSATISFIABLE")


---
**Cell 10 - Reading the answer straight off the unitary (no shots).**

Cell 09 needed 1024 shots to *observe* that the output is always 0.
But we can prove it with certainty by looking at the circuit's math directly,
without running it even once.

**The whole circuit is one unitary.**
Everything before the `measure` - the three $H$ gates and the tree of
Toffoli and $X$ gates - is a single $256 \times 256$ unitary matrix $U$
(there are $2^8 = 256$ basis states for 8 qubits).
The input is *always* the fixed state $|00000000\rangle$, so the output
state the circuit prepares is

$$|\psi\rangle = U\,|00000000\rangle,$$

which is exactly **column 0** of $U$.
Every entry of that column is an exact amplitude, computed algebraically -
there is no sampling and no statistical error.

**Where to look for a "1".**
In Qiskit's little-endian convention qubit $i$ is bit $i$ of the basis
index, so the output qubit $q_7$ is the most significant bit.
A basis state produces a measured 1 on $q_7$ exactly when its index has
bit 7 set (indices 128 through 255).
So the question "can this circuit ever output 1?" becomes a simple search:

> Does column 0 contain **any** nonzero amplitude in a row whose $q_7$ bit is 1?

- If **even one** such amplitude is nonzero, the circuit *can* output 1,
  so the formula is satisfiable.
- If **every** one of those amplitudes is exactly 0, outputting 1 is
  structurally impossible and the formula is unsatisfiable - proven with
  certainty, not estimated from shots.

The cell below extracts $U$, takes column 0, and searches the $q_7 = 1$
rows for any surviving amplitude.
For this unsatisfiable formula the search comes up empty.

In [ ]:
# Cell 10 - Prove the result directly from the circuit's unitary matrix (no shots)
# The circuit before measurement is a single 256x256 unitary U.
# The input is always |00000000>, so the output state is U|0...0>, which is
# exactly column 0 of U. We search that column for any basis state with q7 = 1
# and nonzero amplitude: if even one exists, the circuit CAN output 1; if none
# do, outputting 1 is structurally impossible.

from qiskit.quantum_info import Operator

# Operator needs a purely unitary circuit, so rebuild the Cell 09 circuit
# without its final measurement (measurement is not a unitary operation)
qc_unitary = qc.remove_final_measurements(inplace=False)

U = Operator(qc_unitary).data  # 256 x 256 complex unitary matrix
output_state = U[:, 0]  # U applied to |00000000> = column 0 of U

# q7 is the most significant bit (Qiskit little-endian: qubit i = bit i).
# Search every basis state whose q7 bit is 1 for a surviving amplitude.
tol = 1e-9
producible_ones = []
for index, amplitude in enumerate(output_state):
    q7 = (index >> 7) & 1
    if q7 == 1 and abs(amplitude) > tol:
        bits = format(index, "08b")  # reads q7 q6 q5 q4 q3 q2 q1 q0
        producible_ones.append((bits, amplitude))

print(f"Basis states with q7 = 1 : {2**7}")
print(f"...with nonzero amplitude : {len(producible_ones)}")
print()
if producible_ones:
    print("The circuit CAN output 1 -> formula is SATISFIABLE")
    for bits, amplitude in producible_ones:
        prob = abs(amplitude) ** 2
        print(f"  |{bits}>  amplitude={amplitude.real:+.4f}  probability={prob:.4f}")
else:
    print("No q7 = 1 state carries any amplitude.")
    print("P(q7 = 1) = 0 exactly -> formula is UNSATISFIABLE with certainty")


---
**Cell 11 - Why the unitary "knows" the answer but will not hand it over cheaply.**

Cell 10 is satisfying, but it hides a trap worth making explicit.
Whether $q_7$ can ever read out a 1 is *fully governed* by the circuit's
final unitary. That property is deterministic: there is nothing random
about whether $q_7$ **can** be 1. So it is tempting to conclude that we
should be able to interrogate the unitary directly and skip the shots.

The subtlety is that **"the answer lives in the unitary"** and
**"you can extract the answer cheaply"** are different claims, and the
gap between them is essentially the whole $P$ vs $NP$ question.
Extracting the answer costs one of two exponential prices:

1. **Build the matrix (what Cell 10 does).**
   The circuit is a small, polynomial-size list of gates, but the unitary
   it *denotes* is $2^{(n+\text{ancilla})}$ on a side. Column 0 alone has
   $2^n$ meaningful entries, and searching them for a surviving
   $q_7=1$ amplitude is $2^n$ work. Cell 10 does not avoid brute force;
   it rewrites brute force as a matrix column. For $n=3$ that is 8 rows
   and instant; for $n=64$ it is $2^{64}$ rows and hopeless.

2. **Reason about the gate list directly.**
   You have something more compact than the matrix: the poly-size circuit
   itself. But "can this circuit ever output 1 on $q_7$?" is *literally*
   SAT, and "is $q_7$ always 0?" is *literally* UNSAT (coNP-complete).
   Deciding it by inspecting the circuit is not a shortcut around the hard
   problem: it **is** the hard problem. A polynomial algorithm for it
   would prove $P = NP$.

**This particular circuit makes the point stark: there is no interference.**
The circuit is Hadamards followed by nothing but Toffoli and $X$ gates,
which is pure reversible classical logic. Each of the $2^n$ branches of
the superposition computes $f$ into $q_7$ and keeps distinct values on the
input and ancilla qubits, so the branches stay orthogonal and never
recombine. No amplitudes cancel. The output probability is therefore exact
and interference-free:

$$P(q_7 = 1) = \frac{\text{number of satisfying assignments}}{2^n}$$

Column 0 of the unitary is just the $2^n$ branches sitting side by side,
each carrying its own value of $f$. "Can $q_7$ be 1?" reduces exactly to
"does any branch satisfy $f$?" - the original enumeration. The unitary
governs the answer, but it does so by *storing all $2^n$ branch results
separately*: brute force in disguise.

**The one door that is actually open.**
The only way physics lets a quantum computer beat enumeration is to build
a *different* circuit that uses genuine **interference** - amplitudes that
add and cancel so the wanted outcome concentrates onto a few
high-probability results. Grover's algorithm does exactly this for the SAT
(find-a-witness) side and buys a quadratic speedup, roughly $\sqrt{2^n}$
instead of $2^n$. It still does not certify an exact zero for UNSAT, and
quadratic is not the exponential collapse a one-shot UNSAT proof would
require. A circuit that could interfere its way to certifying exact-zero
efficiently would place coNP inside BQP, which is not believed to be true.

**Bottom line.** The unitary determines whether $q_7$ can read out a 1,
but making that determination *legible* costs either exponential matrix
work or solving the coNP-complete problem itself. Superposition gets the
answer **into** the state; it does not get it **out**.</cell>
